In [1]:
import pandas as pd

crimes = pd.read_csv("Crimes_-_2001_to_Present_20261104.csv")
crime_dates = pd.to_datetime(crimes["Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
cleaned_crimes = crimes.loc[crime_dates.dt.year.eq(2025), ["Date", "Primary Type"]]
cleaned_crimes.to_csv("crimes_cleaned.csv", index=False)

cleaned_crimes.head()

,Date,Primary Type
0,12/31/2025 11:54:00 PM,BATTERY
1,12/31/2025 11:44:00 PM,BATTERY
2,12/31/2025 11:25:00 PM,BATTERY
3,12/31/2025 10:57:00 PM,ASSAULT
4,12/31/2025 10:43:00 PM,BATTERY


In [3]:
import pandas as pd

crimes = pd.read_csv("crimes_cleaned.csv")
weather = pd.read_csv("weather.csv")

crimes["Date"] = pd.to_datetime(
    crimes["Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce"
).dt.normalize()
weather["DATE"] = pd.to_datetime(weather["DATE"], errors="coerce").dt.normalize()

# Restrict both datasets to the requested 2025 analysis year.
crimes = crimes[crimes["Date"].dt.year.eq(2025)]
weather = weather[weather["DATE"].dt.year.eq(2025)].copy()

# Count assault and battery incidents separately for each day.
assault_battery = crimes[crimes["Primary Type"].isin(["ASSAULT", "BATTERY"])]
daily_crimes = (
    assault_battery.dropna(subset=["Date"])
    .groupby(["Date", "Primary Type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["ASSAULT", "BATTERY"], fill_value=0)
    .rename(columns={"ASSAULT": "Assault Count", "BATTERY": "Battery Count"})
    .reset_index()
)

# Average station reports for precipitation and daily high/low midpoint.
for column in ["PRCP", "TMAX", "TMIN"]:
    weather[column] = pd.to_numeric(weather[column], errors="coerce")
weather["Temperature"] = ((weather["TMAX"] + weather["TMIN"]) / 2).where(
    weather["TMAX"].notna() & weather["TMIN"].notna()
)
daily_weather = (
    weather.dropna(subset=["DATE"])
    .groupby("DATE")
    .agg(Precipitation=("PRCP", "mean"), Temperature=("Temperature", "mean"))
    .reset_index()
    .rename(columns={"DATE": "Date"})
)

# Keep all calendar days so days with no reported incidents have a count of zero.
date_range = pd.date_range("2025-01-01", "2025-12-31", freq="D", name="Date")
daily_crimes_weather = (
    pd.DataFrame({"Date": date_range})
    .merge(daily_crimes, on="Date", how="left")
    .merge(daily_weather, on="Date", how="left")
)
count_columns = ["Assault Count", "Battery Count"]
daily_crimes_weather[count_columns] = daily_crimes_weather[count_columns].fillna(0).astype(int)
daily_crimes_weather.to_csv("daily_crimes_weather.csv", index=False)

print("Analysis period:", daily_crimes_weather["Date"].min().date(), "to", daily_crimes_weather["Date"].max().date())
daily_crimes_weather.head()

Analysis period: 2025-01-01 to 2025-12-31


,Date,Assault Count,Battery Count,Precipitation,Temperature
0,2025-01-01,5,35,0.210374,30.250
1,2025-01-02,8,21,0.000467,27.500
2,2025-01-03,2,14,0.010741,22.500
3,2025-01-04,3,20,0.000654,18.250
4,2025-01-05,11,22,0.000180,20.125
